# `c02_adm` — Admissions and Test Scores

**Component curation notebook.** Fetches the raw IPEDS distribution files, verifies the
reference period against official documentation, locks the schema, reshapes to the
declared grain, validates, and writes one curated table with a metadata sidecar.

| Property | Value |
|---|---|
| Native tables | `ADM2023` |
| Reference period | Fall 2023 cohort of first-time degree-seeking undergraduates |
| Curated grain | `UNITID` |
| Output | `data/curated/c02_adm.parquet` |

The funnel identity applications >= admissions >= enrolled is a free integrity check that catches transcription errors and mis-joined years immediately.

> **Pitfall.** Score percentiles are conditional on the submitting subgroup, whose size is `SATPCT`/`ACTPCT`. In a test-optional era those percentages fall sharply and the reported percentiles rise, which is a composition change and not an improvement in selectivity. Never model a score band without carrying its submission share alongside it.

## 1. Environment

One import surface, so a parsing quirk is fixed once rather than twelve times.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import ipeds_utils as iu

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

SLUG = "c02_adm"
TABLES = ['ADM2023']
GRAIN = ['UNITID']
REFERENCE_PERIOD = 'Fall 2023 cohort of first-time degree-seeking undergraduates'

print("ipeds_utils", iu.__version__, "| pandas", pd.__version__)

ipeds_utils 1.1.0 | pandas 3.0.5


## 2. Retrieve

Downloads are cached, so re-running this notebook is offline and cheap. Every retrieval returns a provenance record carrying a SHA-256 digest, which is what makes a result reproducible rather than merely repeatable.

In [2]:
RAW_DIR = "../data/raw"   # relative to notebooks/, so all twelve share one cache

provenance = [iu.fetch(t, raw_dir=RAW_DIR) for t in TABLES]
pd.DataFrame(provenance)[["table", "data_bytes", "data_sha256", "retrieved_utc"]]

,table,data_bytes,data_sha256,retrieved_utc
0,ADM2023,97321,670ecc7c4313f044dcdf740a42ef10c6d2927eb9f890fb...,2026-09-24T17:19:11+00:00


## 3. Verify the reference period

**Do not skip this cell.** The filename year is not the reference period, and the offsets are not uniform across components. This assertion fails loudly rather than letting a misaligned period corrupt every downstream year comparison, where it would be invisible in the data itself.

In [3]:
intro = iu.assert_reference_period(
    provenance[0]["dict_path"],
    expect=r'(fall 2023|2023-24)',
    table=TABLES[0],
)
print(intro[:600])

File Documentation for the Admissions Data File, 2023-24
(Provisional release)
Filename ADM2023
Overview This file contains information about the undergraduate selection process for entering first-time, degree/certificate-seeking students. This includes information about admission considerations,  applicants, applicants that were admitted, and admitted students who enrolled. SAT and ACT test scores are included for institutions, that require or consider test scores for admission. These data are applicable for institutions that do not have an open admissions policy for entering first-time stude


## 4. Inspect the dictionary

Variable labels come from the published dictionary, never from memory. This is also where value sets are read, so categorical decoding is driven by the official codebook and a taxonomy revision surfaces as unmatched codes instead of a plausible-looking wrong label.

In [4]:
variables = iu.read_dict(provenance[0]["dict_path"])
valuesets = iu.read_valuesets(provenance[0]["dict_path"])

print(f"{len(variables)} variables documented, {len(valuesets)} value-set rows")
variables[["varname", "vartitle"]].head(20)

57 variables documented, 35 value-set rows


,varname,vartitle
0,UNITID,Unique identification number of the institution
1,ADMCON1,Secondary school GPA
2,ADMCON2,Secondary school rank
3,ADMCON3,Secondary school record
4,ADMCON4,Completion of college-preparatory program
5,ADMCON5,Recommendations
6,ADMCON6,Formal demonstration of competencies
7,ADMCON7,Admission test scores
8,ADMCON8,English Proficiency Test
9,ADMCON9,"Other Test (Wonderlic, WISC-III, etc.)"


## 5. Load and lock the schema

The first run records the column signature; later runs fail if it drifts.

In [5]:
KEEP = ['UNITID', 'APPLCN', 'APPLCNM', 'APPLCNW', 'ADMSSN', 'ADMSSNM', 'ADMSSNW', 'ENRLT', 'ENRLM', 'ENRLW', 'SATNUM', 'SATPCT', 'ACTNUM', 'ACTPCT', 'SATVR25', 'SATVR75', 'SATMT25', 'SATMT75', 'ACTCM25', 'ACTCM75', 'ADMCON1', 'ADMCON2']

raw = iu.read_csv(provenance[0]["data_path"])
print("raw shape", raw.shape)

lock = iu.lock_schema(raw, TABLES[0], schema_dir="../schemas", strict=False)
print("schema:", lock["status"], "| added", lock["added"][:5], "| removed", lock["removed"][:5])

available = [c for c in KEEP if c in raw.columns]
missing = [c for c in KEEP if c not in raw.columns]
if missing:
    print("NOT PRESENT in this cycle (verify against the varlist above):", missing)

frame = raw[available].copy()
frame.head()

raw shape (1972, 101)
schema: unchanged | added [] | removed []


,UNITID,APPLCN,APPLCNM,APPLCNW,ADMSSN,ADMSSNM,ADMSSNW,ENRLT,ENRLM,ENRLW,SATNUM,SATPCT,ACTNUM,ACTPCT,SATVR25,SATVR75,SATMT25,SATMT75,ACTCM25,ACTCM75,ADMCON1,ADMCON2
0,100654,15628,5247,10381,10349.0,3409.0,6940.0,1956.0,799.0,1157.0,462.0,24.0,1312.0,67.0,420.0,540.0,390.0,520.0,14.0,19.0,1,3
1,100663,10919,3973,6918,9655.0,3329.0,6301.0,2095.0,694.0,1398.0,112.0,5.0,1021.0,49.0,570.0,700.0,560.0,700.0,22.0,30.0,1,3
2,100706,6074,2984,3090,4510.0,2374.0,2136.0,1176.0,771.0,405.0,68.0,6.0,686.0,58.0,595.0,700.0,600.0,740.0,25.0,31.0,1,3
3,100724,5346,1810,3490,5113.0,1707.0,3362.0,951.0,353.0,589.0,88.0,9.0,312.0,33.0,444.0,538.0,421.0,531.0,16.0,20.0,1,3
4,100751,58418,23331,35087,44295.0,17444.0,26851.0,8279.0,3520.0,4759.0,1468.0,18.0,3756.0,45.0,590.0,700.0,580.0,700.0,24.0,31.0,1,5


## 6. Mask reserved missing codes

IPEDS encodes missingness as negative integers. A mean computed without masking them is badly wrong and looks entirely plausible, which is what makes this the most costly single omission in IPEDS analysis.

In [6]:
RESERVED = [-1, -2, -3, -9]

numeric_cols = [
    c for c in frame.columns
    if c not in ("UNITID", *GRAIN) and pd.api.types.is_numeric_dtype(frame[c])
]

before = frame[numeric_cols].isna().sum().sum()
for col in numeric_cols:
    frame.loc[frame[col].isin(RESERVED), col] = np.nan
after = frame[numeric_cols].isna().sum().sum()

# Masking turns an integer column into float (1 becomes 1.0). Measures can stay float,
# since NaN is what the models expect, but category codes go back to nullable integers
# so they print, join, and decode as codes rather than as 1.0.
for col in ['ADMCON1', 'ADMCON2']:
    if col in frame.columns and pd.api.types.is_float_dtype(frame[col]):
        if (frame[col].dropna() % 1 == 0).all():
            frame[col] = frame[col].astype("Int64")

print(f"masked {after - before:,} reserved-code cells across {len(numeric_cols)} numeric columns")

masked 0 reserved-code cells across 21 numeric columns


## 7. Carry the imputation flags

An imputed value and a reported value are not the same evidence. A column where most institutions carry a generated flag should not be modelled as though it were observed, and this is where that judgement becomes possible.

In [7]:
values, flags = iu.split_imputation_flags(raw, numeric_cols)

if flags.shape[1] > 1:
    summary = iu.imputation_summary(flags)
    display(summary.head(15))
    reported = summary[summary.flag == "R"].set_index("column")["share"]
    weak = reported[reported < 0.90]
    if len(weak):
        print("Columns under 90% reported — interpret with care:")
        display(weak)
else:
    print("No X-prefixed imputation flags accompany this file.")

,column,flag,n,share
64,XACTCM25,R,972,0.4929
65,XACTCM25,A,928,0.4706
66,XACTCM25,S,71,0.0360
67,XACTCM25,N,1,0.0005
68,XACTCM75,R,972,0.4929
69,XACTCM75,A,928,0.4706
70,XACTCM75,S,71,0.0360
71,XACTCM75,N,1,0.0005
31,XACTNUM,R,1157,0.5867
32,XACTNUM,A,811,0.4113


Columns under 90% reported — interpret with care:


column
XACTCM25    0.4929
XACTCM75    0.4929
XACTNUM     0.5867
XACTPCT     0.5761
XSATMT25    0.5030
XSATMT75    0.5030
XSATNUM     0.5898
XSATPCT     0.5801
XSATVR25    0.5030
XSATVR75    0.5030
Name: share, dtype: float64

## 8. Decode categoricals

Labels from the published value sets, not hand-typed mappings.

In [8]:
CATEGORICALS = ['ADMCON1', 'ADMCON2']

unresolved = {}
for col in CATEGORICALS:
    if col in frame.columns:
        frame = iu.decode(frame, valuesets, col)
        unmatched = frame.loc[frame[col].notna() & frame[f"{col}_LABEL"].isna(), col].unique()
        if len(unmatched):
            unresolved[col] = sorted(unmatched.tolist())[:10]

# An unmatched code means a taxonomy change or a parsing fault. Either way the
# labels are wrong, so this stops the notebook rather than printing a warning.
assert not unresolved, f"codes absent from the published value set: {unresolved}"

label_cols = [c for c in frame.columns if c.endswith("_LABEL")]
frame[CATEGORICALS + label_cols].drop_duplicates().head(20) if label_cols else frame.head()

,ADMCON1,ADMCON2,ADMCON1_LABEL,ADMCON2_LABEL
0,1,3,Required to be considered for admission,"Not considered for admission, even if submitted"
4,1,5,Required to be considered for admission,"Not required for admission, but considered if ..."
11,5,5,"Not required for admission, but considered if ...","Not required for admission, but considered if ..."
27,5,3,"Not required for admission, but considered if ...","Not considered for admission, even if submitted"
53,1,1,Required to be considered for admission,Required to be considered for admission
110,3,3,"Not considered for admission, even if submitted","Not considered for admission, even if submitted"
263,3,5,"Not considered for admission, even if submitted","Not required for admission, but considered if ..."
888,3,1,"Not considered for admission, even if submitted",Required to be considered for admission
1440,5,1,"Not required for admission, but considered if ...",Required to be considered for admission


## 9. Reshape to the declared grain

Target grain: `UNITID`. The grain is asserted, not assumed, because a duplicated key silently inflates every aggregate computed downstream.

In [9]:
curated = frame.copy()

# This component already arrives at its declared grain, so curation is a
# pass-through. Components with a long layout (GRTYPE, EFFYALEV, STAFFCAT,
# OMCHRT) filter or pivot here instead; see c10_f for a full worked reshape.

present_grain = [g for g in GRAIN if g in curated.columns]
duplicated = curated.duplicated(subset=present_grain, keep=False).sum()
print(f"grain {present_grain} -> {len(curated):,} rows, {duplicated} duplicated")
assert duplicated == 0, "Declared grain is not unique; resolve before continuing."

curated.head()

grain ['UNITID'] -> 1,972 rows, 0 duplicated


,UNITID,APPLCN,APPLCNM,APPLCNW,ADMSSN,ADMSSNM,ADMSSNW,ENRLT,ENRLM,ENRLW,SATNUM,SATPCT,ACTNUM,ACTPCT,SATVR25,SATVR75,SATMT25,SATMT75,ACTCM25,ACTCM75,ADMCON1,ADMCON2,ADMCON1_LABEL,ADMCON2_LABEL
0,100654,15628.0,5247.0,10381.0,10349.0,3409.0,6940.0,1956.0,799.0,1157.0,462.0,24.0,1312.0,67.0,420.0,540.0,390.0,520.0,14.0,19.0,1,3,Required to be considered for admission,"Not considered for admission, even if submitted"
1,100663,10919.0,3973.0,6918.0,9655.0,3329.0,6301.0,2095.0,694.0,1398.0,112.0,5.0,1021.0,49.0,570.0,700.0,560.0,700.0,22.0,30.0,1,3,Required to be considered for admission,"Not considered for admission, even if submitted"
2,100706,6074.0,2984.0,3090.0,4510.0,2374.0,2136.0,1176.0,771.0,405.0,68.0,6.0,686.0,58.0,595.0,700.0,600.0,740.0,25.0,31.0,1,3,Required to be considered for admission,"Not considered for admission, even if submitted"
3,100724,5346.0,1810.0,3490.0,5113.0,1707.0,3362.0,951.0,353.0,589.0,88.0,9.0,312.0,33.0,444.0,538.0,421.0,531.0,16.0,20.0,1,3,Required to be considered for admission,"Not considered for admission, even if submitted"
4,100751,58418.0,23331.0,35087.0,44295.0,17444.0,26851.0,8279.0,3520.0,4759.0,1468.0,18.0,3756.0,45.0,590.0,700.0,580.0,700.0,24.0,31.0,1,5,Required to be considered for admission,"Not required for admission, but considered if ..."


## 10. Validate

Rules are declarative so the output is a persistable report: which checks ran, which failed, on how many rows, and which institutions were implicated. That report is the artefact you cite when claiming this table is fit for analysis.

In [10]:
RULES = [
    iu.unique_key('UNITID'),
    iu.in_range('APPLCN', 0, None),
    iu.Rule('admitted_le_applied', lambda d: pd.to_numeric(d.ADMSSN, errors='coerce') > pd.to_numeric(d.APPLCN, errors='coerce'), note='Admits cannot exceed applicants'),
    iu.Rule('enrolled_le_admitted', lambda d: pd.to_numeric(d.ENRLT, errors='coerce') > pd.to_numeric(d.ADMSSN, errors='coerce'), note='Enrolled cannot exceed admits'),
]

report = iu.validate(curated, RULES, SLUG)
report.save(f"../reports/validation/{SLUG}.json")
display(report.to_frame()[["name", "status", "n_offending", "share", "note"]])

print("PASSED" if report.ok else "FAILED")
report.raise_if_failed()

,name,status,n_offending,share,note
0,unique_key(UNITID),pass,0,0.0,Declared grain must be unique
1,"in_range(APPLCN,0,None)",pass,0,0.0,Value plausibility bound
2,admitted_le_applied,pass,0,0.0,Admits cannot exceed applicants
3,enrolled_le_admitted,pass,0,0.0,Enrolled cannot exceed admits


PASSED


Report(table='c02_adm', rows=1972, results=[{'name': 'unique_key(UNITID)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Declared grain must be unique', 'status': 'pass'}, {'name': 'in_range(APPLCN,0,None)', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Value plausibility bound', 'status': 'pass'}, {'name': 'admitted_le_applied', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Admits cannot exceed applicants', 'status': 'pass'}, {'name': 'enrolled_le_admitted', 'severity': 'error', 'n_offending': 0, 'share': 0.0, 'sample_unitids': [], 'note': 'Enrolled cannot exceed admits', 'status': 'pass'}], generated_utc='2026-09-24T17:19:11+00:00')

## 11. Write the curated table

The sidecar carries the reference period and grain with the data. This is the defence against assembling a panel by filename year when the underlying periods are offset differently per component.

In [11]:
path = iu.write_curated(
    curated,
    SLUG,
    root="../data/curated",
    reference_period=REFERENCE_PERIOD,
    grain=GRAIN,
    provenance=provenance,
    notes='Score percentiles are conditional on the submitting subgroup, whose size is `SATPCT`/`ACTPCT`. In a test-optional era those percentages fall sharply and the reported percentiles rise, which is a composition change and not an improvement in selectivity. Never model a score band without carrying its submission share alongside it.',
)

iu.write_provenance(provenance, f"../docs/provenance/{SLUG}.json")
print("wrote", path, f"({len(curated):,} rows x {curated.shape[1]} columns)")

wrote ../data/curated/c02_adm.parquet (1,972 rows x 24 columns)


## 12. Exercises

1. Re-run this notebook against the prior collection cycle by changing `TABLES`. The schema lock and the period assertion will both object; resolve each objection and record what changed between cycles.
2. Identify the three columns with the lowest reported-flag share, and argue whether each belongs in a predictive model at all.
3. Construct one derived cross-tabulation from this table, then apply `iu.suppress` and `iu.k_anonymity` to it. Report the smallest equivalence class before and after coarsening, and state the k you would require before publishing.
4. Score percentiles are conditional on the submitting subgroup, whose size is `SATPCT`/`ACTPCT`. Write a validation rule that would catch this error if a colleague made it, and add it to `RULES` above.